# 00 — Tải dữ liệu
Chỉ phân tích các shard đã tải; thứ tự shard không bảo đảm mẫu đại diện. Revision được chốt theo commit khi tải. Tăng số shard nếu cần phủ thời gian rộng hơn.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
from spark_common import create_spark_session
from project_config import DATA_DIR, RESULTS_DIR, OUTPUT_DIR
from huggingface_hub import HfApi, hf_hub_download
import hashlib, shutil, json
from batch_pipeline import save_json
api = HfApi()
repo = "VatsaDev/test_arxiv"
revision = api.repo_info(repo, repo_type="dataset", revision="refs/convert/parquet").sha
files = sorted([f for f in api.list_repo_tree(repo, path_in_repo="default/train", repo_type="dataset", revision=revision)
                if getattr(f, "path", "").endswith(".parquet")], key=lambda f: f.path)
print("Available shards:", [(f.path, f.size) for f in files])


In [ ]:
NUM_SHARDS_TO_DOWNLOAD = 2
if not 1 <= NUM_SHARDS_TO_DOWNLOAD <= len(files):
    raise ValueError("Invalid shard count")
target = DATA_DIR / "arxiv_data.parquet"
target.mkdir(parents=True, exist_ok=True)
selected = files[:NUM_SHARDS_TO_DOWNLOAD]
expected = {Path(f.path).name for f in selected}
existing = {p.name for p in target.glob("*.parquet")}
if existing - expected:
    raise ValueError("Thư mục có shard ngoài lựa chọn; giữ nguyên dữ liệu và điều chỉnh lựa chọn trước khi tải")
for entry in selected:
    # Download/cache is keyed by the immutable repository revision.
    downloaded = hf_hub_download(repo, filename=entry.path, repo_type="dataset", revision=revision)
    shutil.copy2(downloaded, target / Path(entry.path).name)
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
shards = []
for entry in selected:
    local = target / Path(entry.path).name
    shards.append(dict(path=entry.path, local_name=local.name, bytes=local.stat().st_size, sha256=sha256(local)))
save_json(DATA_DIR / "source_manifest.json", dict(repo=repo, revision=revision, revision_ref="refs/convert/parquet",
          shards=shards, total_bytes=sum(item["bytes"] for item in shards), scope="downloaded_shards_only",
          note="Chỉ phân tích các shard đã tải; không đại diện toàn bộ arXiv."))
print("Downloaded", len(selected), "shards at", revision)
